In [5]:
from pyspark.sql import functions as F

df = spark.read.table("silver_flipkart_products")

text = F.lower(F.concat_ws(" ", F.col("product_name"), F.col("description"), F.col("main_category")))

target_customer = (
    F.when(text.rlike("baby|kid|child|toy|school"), "Kids")
     .when(text.rlike("women|woman|girl|ladies"), "Women")
     .when(text.rlike("\\bmen\\b|\\bman\\b|\\bboy"), "Men")
     .when(text.rlike("home|kitchen|furniture|decor"), "Home")
     .when(text.rlike("laptop|office|professional|printer"), "Professional")
     .otherwise("Unisex")
)

desc_quality = (
    F.when(F.col("description_length") >= 200, "Excellent")
     .when(F.col("description_length") >= 80, "Good")
     .otherwise("Poor")
)

quality_score = (
    F.when(F.col("description_length") >= 200, 4).when(F.col("description_length") >= 80, 3).otherwise(1)
    + F.when(F.col("brand").isNotNull() & (F.length(F.trim(F.col("brand"))) > 0), 2).otherwise(0)
    + F.when(F.col("discounted_price") > 0, 2).otherwise(0)
    + F.when(F.col("discount_percentage").between(5, 70), 2).otherwise(1)
)

name_words = F.split(F.trim(F.regexp_replace(F.lower(F.col("product_name")), "[^a-z0-9 ]", " ")), "\\s+")
key_words = F.filter(
    name_words,
    lambda w: (F.length(w) >= 4) & (~w.isin("with", "from", "this", "that", "pack", "flipkart", "product"))
)
key_features = F.array_join(F.slice(key_words, 1, 3), " ")

df_ai = (
    df.filter(F.col("description").isNotNull())
      .filter(F.col("description_length") > 50)
      .select(
          "product_id", "product_name", "description", "main_category",
          "retail_price", "discounted_price", "discount_percentage",
          "price_category", "brand",
          F.least(quality_score, F.lit(10)).cast("int").alias("ai_quality_score"),
          F.when(F.length(key_features) > 0, key_features).otherwise("Standard Product").alias("ai_key_features"),
          target_customer.alias("ai_target_customer"),
          desc_quality.alias("ai_description_quality")
      )
      .limit(500)
)

df_ai.write.format("delta").mode("overwrite").saveAsTable("gold_flipkart_ai_products")
print("Saved gold_flipkart_ai_products")
df_ai.groupBy("ai_target_customer").count().orderBy(F.desc("count")).show()

StatementMeta(, 37d658dd-2b91-4d32-8e29-321c05da5dae, 14, Finished, Available, Finished, False)

Saved gold_flipkart_ai_products
+------------------+-----+
|ai_target_customer|count|
+------------------+-----+
|             Women|  190|
|            Unisex|  169|
|               Men|  101|
|              Home|   25|
|              Kids|   12|
|      Professional|    3|
+------------------+-----+

